In [0]:
%sql
WITH ultima_versao AS (SELECT nct_id,
                              payload,
                              ROW_NUMBER() OVER (PARTITION BY nct_id ORDER BY collected_at DESC, page_number DESC) AS rn
                         FROM mvp_eng_dados.mvp_cancer.brz_clinical_trials),

         registros AS (SELECT nct_id,
                              TRY_CAST(get_json_object(payload,'$.protocolSection.statusModule.studyFirstPostDateStruct.date') AS DATE) AS data_primeira_publicacao
                         FROM ultima_versao
                        WHERE rn = 1)
                        
SELECT YEAR(data_primeira_publicacao) AS ano_registro,
       COUNT(DISTINCT nct_id) AS quantidade_estudos
  FROM registros
 GROUP BY YEAR(data_primeira_publicacao)
 ORDER BY ano_registro NULLS LAST;

In [0]:
%sql
SELECT country_name AS pais,
       country_iso3 AS codigo_iso3,
       CASE WHEN country_iso3 IS NULL THEN 'SEM_MAPEAMENTO' ELSE 'MAPEADO' END AS situacao_mapeamento,
       COUNT(DISTINCT nct_id) AS quantidade_estudos,
       COUNT(*) AS registros_localizacao
  FROM mvp_eng_dados.mvp_cancer.slv_locations_iso3
 GROUP BY country_name, country_iso3
 ORDER BY quantidade_estudos DESC, pais;

In [0]:
%sql
WITH estudos_por_pais AS (SELECT c.country_iso3,
                                 c.country_name,
                                 COUNT(DISTINCT b.study_key) AS quantidade_estudos
                            FROM mvp_eng_dados.mvp_cancer.gld_flat_bridge_study_location b
                            JOIN mvp_eng_dados.mvp_cancer.gld_dim_country c
                                ON b.country_key = c.country_key
                            GROUP BY c.country_iso3, c.country_name)

SELECT DENSE_RANK() OVER (ORDER BY quantidade_estudos DESC) AS posicao,
       country_iso3 AS codigo_iso3,
       country_name AS pais,
       quantidade_estudos
  FROM estudos_por_pais
 ORDER BY posicao, pais;